In [2]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor

In [3]:
train = pd.read_pickle("../data/processed/train.pkl")
validation = pd.read_pickle("../data/processed/validation.pkl")
test = pd.read_pickle("../data/processed/test.pkl")

In [4]:
train.shape, validation.shape, test.shape

((473436, 15), (118664, 15), (121160, 15))

In [5]:
model_features = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_std_7",
    "rolling_mean_28",
    "day_of_week",
    "month",
    "week_of_year",
    "is_weekend",
]

In [7]:
X_train = train[model_features]
y_train = train["target"]

X_validation = validation[model_features]
y_validation = validation["target"]

In [8]:
print(X_train.shape)
print(X_validation.shape)

(473436, 11)
(118664, 11)


In [3]:
validation["naive_prediction"] = validation["lag_1"]

In [4]:
validation[["StockCode", "Date", "Demand", "lag_1", "naive_prediction", "target"]].head(
    15
)

,StockCode,Date,Demand,lag_1,naive_prediction,target
1036,10125,2011-08-20,0,20.0,20.0,0.0
1037,10125,2011-08-21,0,0.0,0.0,0.0
1038,10125,2011-08-22,0,0.0,0.0,0.0
1039,10125,2011-08-23,0,0.0,0.0,25.0
1040,10125,2011-08-24,25,0.0,0.0,0.0
1041,10125,2011-08-25,0,25.0,25.0,0.0
1042,10125,2011-08-26,0,0.0,0.0,0.0
1043,10125,2011-08-27,0,0.0,0.0,0.0
1044,10125,2011-08-28,0,0.0,0.0,0.0
1045,10125,2011-08-29,0,0.0,0.0,0.0


In [6]:
from sklearn.metrics import mean_absolute_error

mae_naive = mean_absolute_error(validation["target"], validation["naive_prediction"])

mae_naive

11.302534888424459

In [7]:
from sklearn.metrics import mean_squared_error

rmse_naive = (
    mean_squared_error(validation["target"], validation["naive_prediction"]) ** 0.5
)

rmse_naive

49.119678713502225

In [8]:
zero_actual = validation["target"] == 0

zero_prediction_correct = (validation.loc[zero_actual, "naive_prediction"] == 0).mean()

zero_prediction_correct

np.float64(0.7388078031879823)

In [10]:
baseline_data = (
    pd.concat([train, validation, test], ignore_index=True)
    .sort_values(["StockCode", "Date"])
    .copy()
)

In [11]:
baseline_data["ma_7"] = baseline_data.groupby("StockCode")["Demand"].transform(
    lambda x: x.rolling(7).mean()
)

In [12]:
baseline_data[["StockCode", "Date", "Demand", "ma_7"]].head(15)

,StockCode,Date,Demand,ma_7
0,10002,2010-12-29,0,NaN
1,10002,2010-12-30,0,NaN
2,10002,2010-12-31,0,NaN
3,10002,2011-01-01,0,NaN
4,10002,2011-01-02,0,NaN
5,10002,2011-01-03,0,NaN
6,10002,2011-01-04,0,0.000000
7,10002,2011-01-05,12,1.714286
8,10002,2011-01-06,60,10.285714
9,10002,2011-01-07,1,10.428571


In [13]:
baseline_data[["StockCode", "Date", "Demand", "ma_7", "target"]].head(15)

,StockCode,Date,Demand,ma_7,target
0,10002,2010-12-29,0,NaN,0.0
1,10002,2010-12-30,0,NaN,0.0
2,10002,2010-12-31,0,NaN,0.0
3,10002,2011-01-01,0,NaN,0.0
4,10002,2011-01-02,0,NaN,0.0
5,10002,2011-01-03,0,NaN,0.0
6,10002,2011-01-04,0,0.000000,12.0
7,10002,2011-01-05,12,1.714286,60.0
8,10002,2011-01-06,60,10.285714,1.0
9,10002,2011-01-07,1,10.428571,0.0


In [14]:
validation_ma = baseline_data[
    (baseline_data["Date"] > "2011-08-19") & (baseline_data["Date"] <= "2011-10-12")
].copy()

validation_ma = validation_ma.dropna(subset=["ma_7", "target"])

In [15]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae_ma7 = mean_absolute_error(validation_ma["target"], validation_ma["ma_7"])

rmse_ma7 = mean_squared_error(validation_ma["target"], validation_ma["ma_7"]) ** 0.5

print("7-Day Moving Average MAE:", mae_ma7)
print("7-Day Moving Average RMSE:", rmse_ma7)

7-Day Moving Average MAE: 9.553136212849502
7-Day Moving Average RMSE: 37.12059034442131


In [17]:
model_features = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_std_7",
    "rolling_mean_28",
    "day_of_week",
    "month",
    "week_of_year",
    "is_weekend",
]

In [18]:
from sklearn.linear_model import LinearRegression

X_train = train[model_features]
y_train = train["target"]

X_validation = validation[model_features]
y_validation = validation["target"]

In [19]:
linear_model = LinearRegression()

linear_model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](11,)","[-0.02, 0.01, 0.01,..., 0.3 ,-0.09, 3.78]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](11,)","['lag_1','lag_7','lag_14',...,'month','week_of_year','is_weekend']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,5.119
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,11
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,11


In [20]:
linear_validation_pred = linear_model.predict(X_validation)

In [21]:
comparison = pd.DataFrame(
    {"Actual": y_validation.values, "Predicted": linear_validation_pred}
)

comparison.head(10)

,Actual,Predicted
0,0.0,1.558807
1,0.0,0.386629
2,0.0,5.445637
3,25.0,3.957255
4,0.0,2.468872
5,0.0,2.719063
6,0.0,1.579398
7,0.0,2.223451
8,0.0,0.735069
9,0.0,5.794077


In [22]:
print("Min prediction:", linear_validation_pred.min())
print("Max prediction:", linear_validation_pred.max())
print("Negative predictions:", (linear_validation_pred < 0).sum())

Min prediction: -7.820641066170015
Max prediction: 256.7697919950221
Negative predictions: 14854


In [23]:
mae_linear = mean_absolute_error(y_validation, linear_validation_pred)

rmse_linear = mean_squared_error(y_validation, linear_validation_pred) ** 0.5

print("Linear Regression MAE:", mae_linear)
print("Linear Regression RMSE:", rmse_linear)

Linear Regression MAE: 8.798927561624463
Linear Regression RMSE: 35.23593393973345


In [24]:
coefficients = pd.DataFrame(
    {"Feature": model_features, "Coefficient": linear_model.coef_}
).sort_values("Coefficient", ascending=False)

coefficients

,Feature,Coefficient
10,is_weekend,3.781816
4,rolling_mean_7,0.681176
6,rolling_mean_28,0.539987
8,month,0.298514
1,lag_7,0.007967
2,lag_14,0.005147
0,lag_1,-0.015810
3,lag_28,-0.016340
9,week_of_year,-0.089469
5,rolling_std_7,-0.280860


In [26]:
import xgboost as xgb

print(xgb.__version__)

3.2.0


In [27]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

In [28]:
xgb_model.fit(X_train, y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [29]:
xgb_validation_pred = xgb_model.predict(X_validation)

In [30]:
xgb_comparison = pd.DataFrame(
    {"Actual": y_validation.values, "Predicted": xgb_validation_pred}
)

xgb_comparison.head(10)

,Actual,Predicted
0,0.0,2.456470
1,0.0,3.223011
2,0.0,3.974154
3,25.0,3.977347
4,0.0,5.490889
5,0.0,5.102201
6,0.0,-0.103185
7,0.0,2.564731
8,0.0,4.480474
9,0.0,5.216669


In [31]:
print("Min prediction:", xgb_validation_pred.min())
print("Max prediction:", xgb_validation_pred.max())
print("Negative predictions:", (xgb_validation_pred < 0).sum())

Min prediction: -63.571396
Max prediction: 612.36365
Negative predictions: 4668


In [32]:
mae_xgb = mean_absolute_error(y_validation, xgb_validation_pred)

rmse_xgb = mean_squared_error(y_validation, xgb_validation_pred) ** 0.5

print("XGBoost MAE:", mae_xgb)
print("XGBoost RMSE:", rmse_xgb)

XGBoost MAE: 8.675399956501595
XGBoost RMSE: 35.58789610685837


In [33]:
xgb_results = pd.DataFrame(
    {"Actual": y_validation.values, "Predicted": xgb_validation_pred}
)

xgb_results["Error"] = xgb_results["Actual"] - xgb_results["Predicted"]

xgb_results["Absolute_Error"] = xgb_results["Error"].abs()

In [34]:
print(
    "Zero-demand MAE:",
    xgb_results.loc[xgb_results["Actual"] == 0, "Absolute_Error"].mean(),
)

print(
    "Non-zero demand MAE:",
    xgb_results.loc[xgb_results["Actual"] > 0, "Absolute_Error"].mean(),
)

Zero-demand MAE: 3.5264963060206465
Non-zero demand MAE: 18.31649177407204


In [35]:
high_demand = xgb_results[xgb_results["Actual"] >= 20]

print("High-demand observations:", len(high_demand))
print("High-demand MAE:", high_demand["Absolute_Error"].mean())
print("High-demand RMSE:", (high_demand["Error"] ** 2).mean() ** 0.5)

High-demand observations: 11782
High-demand MAE: 47.00839356266998
High-demand RMSE: 108.85909611036378


In [36]:
linear_results = pd.DataFrame(
    {"Actual": y_validation.values, "Predicted": linear_validation_pred}
)

linear_results["Error"] = linear_results["Actual"] - linear_results["Predicted"]

linear_results["Absolute_Error"] = linear_results["Error"].abs()

linear_high_demand = linear_results[linear_results["Actual"] >= 20]

print("High-demand observations:", len(linear_high_demand))
print("High-demand MAE:", linear_high_demand["Absolute_Error"].mean())
print("High-demand RMSE:", (linear_high_demand["Error"] ** 2).mean() ** 0.5)

High-demand observations: 11782
High-demand MAE: 46.16075144016594
High-demand RMSE: 108.76031881332261


In [37]:
zero_prediction = pd.Series(0, index=y_validation.index)

zero_mae = mean_absolute_error(y_validation, zero_prediction)

zero_rmse = mean_squared_error(y_validation, zero_prediction) ** 0.5

print("Zero Baseline MAE:", zero_mae)
print("Zero Baseline RMSE:", zero_rmse)

Zero Baseline MAE: 7.705673161194634
Zero Baseline RMSE: 37.840731405191534


In [38]:
import numpy as np

y_train_log = np.log1p(y_train)
y_validation_log = np.log1p(y_validation)

print("Original target:")
print(y_train.describe())

print("\nLog-transformed target:")
print(y_train_log.describe())

Original target:
count    473436.000000
mean          5.402054
std          32.818478
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max        4314.000000
Name: target, dtype: float64

Log-transformed target:
count    473436.000000
mean          0.605272
std           1.143955
min           0.000000
25%           0.000000
50%           0.000000
75%           0.693147
max           8.369853
Name: target, dtype: float64


In [39]:
xgb_log_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

In [40]:
xgb_log_model.fit(X_train, y_train_log)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [41]:
xgb_log_validation_pred = xgb_log_model.predict(X_validation)

In [42]:
xgb_log_validation_pred = np.expm1(xgb_log_validation_pred)

In [43]:
log_xgb_comparison = pd.DataFrame(
    {"Actual": y_validation.values, "Predicted": xgb_log_validation_pred}
)

log_xgb_comparison.head(10)

,Actual,Predicted
0,0.0,0.271964
1,0.0,0.735994
2,0.0,0.601238
3,25.0,0.913710
4,0.0,1.109573
5,0.0,1.047087
6,0.0,-0.085015
7,0.0,0.292099
8,0.0,0.657254
9,0.0,0.556596


In [44]:
print("Min prediction:", xgb_log_validation_pred.min())
print("Max prediction:", xgb_log_validation_pred.max())
print("Negative predictions:", (xgb_log_validation_pred < 0).sum())

Min prediction: -0.45445207
Max prediction: 200.98978
Negative predictions: 8301


In [45]:
mae_xgb_log = mean_absolute_error(y_validation, xgb_log_validation_pred)

rmse_xgb_log = mean_squared_error(y_validation, xgb_log_validation_pred) ** 0.5

print("Log-target XGBoost MAE:", mae_xgb_log)
print("Log-target XGBoost RMSE:", rmse_xgb_log)

Log-target XGBoost MAE: 7.0496216218894965
Log-target XGBoost RMSE: 36.04395230652256


In [46]:
log_xgb_results = pd.DataFrame(
    {"Actual": y_validation.values, "Predicted": xgb_log_validation_pred}
)

log_xgb_results["Error"] = log_xgb_results["Actual"] - log_xgb_results["Predicted"]

log_xgb_results["Absolute_Error"] = log_xgb_results["Error"].abs()

log_high_demand = log_xgb_results[log_xgb_results["Actual"] >= 20]

print("High-demand observations:", len(log_high_demand))
print("High-demand MAE:", log_high_demand["Absolute_Error"].mean())
print("High-demand RMSE:", (log_high_demand["Error"] ** 2).mean() ** 0.5)

High-demand observations: 11782
High-demand MAE: 53.84891779552778
High-demand RMSE: 113.73290815974049


In [48]:
xgb_validation_pred_clipped = np.maximum(xgb_validation_pred, 0)

In [49]:
mae_xgb_clipped = mean_absolute_error(y_validation, xgb_validation_pred_clipped)

rmse_xgb_clipped = mean_squared_error(y_validation, xgb_validation_pred_clipped) ** 0.5

print("Clipped XGBoost MAE:", mae_xgb_clipped)
print("Clipped XGBoost RMSE:", rmse_xgb_clipped)

Clipped XGBoost MAE: 8.620960051768932
Clipped XGBoost RMSE: 35.578313935560736


In [50]:
xgb_depth3 = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

In [51]:
xgb_depth3.fit(X_train, y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [52]:
depth3_pred = xgb_depth3.predict(X_validation)

In [53]:
depth3_mae = mean_absolute_error(y_validation, depth3_pred)

depth3_rmse = mean_squared_error(y_validation, depth3_pred) ** 0.5

print("Depth 3 MAE:", depth3_mae)
print("Depth 3 RMSE:", depth3_rmse)

Depth 3 MAE: 8.503683057897073
Depth 3 RMSE: 34.79291801651643


In [54]:
xgb_depth5 = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

In [55]:
xgb_depth5.fit(X_train, y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [56]:
depth5_pred = xgb_depth5.predict(X_validation)

depth5_mae = mean_absolute_error(y_validation, depth5_pred)

depth5_rmse = mean_squared_error(y_validation, depth5_pred) ** 0.5

print("Depth 5 MAE:", depth5_mae)
print("Depth 5 RMSE:", depth5_rmse)

Depth 5 MAE: 8.556121791886474
Depth 5 RMSE: 35.319036377255635


In [57]:
xgb_depth7 = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

xgb_depth7.fit(X_train, y_train)

depth7_pred = xgb_depth7.predict(X_validation)

depth7_mae = mean_absolute_error(y_validation, depth7_pred)

depth7_rmse = mean_squared_error(y_validation, depth7_pred) ** 0.5

print("Depth 7 MAE:", depth7_mae)
print("Depth 7 RMSE:", depth7_rmse)

Depth 7 MAE: 8.842655853654856
Depth 7 RMSE: 35.86338065603863


In [58]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

param_grid = {"learning_rate": [0.10, 0.05, 0.03], "n_estimators": [200, 300, 500]}

results = []

for learning_rate in param_grid["learning_rate"]:
    for n_estimators in param_grid["n_estimators"]:

        model = XGBRegressor(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=3,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)

        predictions = model.predict(X_validation)

        mae = mean_absolute_error(y_validation, predictions)

        rmse = mean_squared_error(y_validation, predictions) ** 0.5

        results.append(
            {
                "learning_rate": learning_rate,
                "n_estimators": n_estimators,
                "MAE": mae,
                "RMSE": rmse,
            }
        )

results_df = pd.DataFrame(results)

results_df.sort_values("RMSE")

,learning_rate,n_estimators,MAE,RMSE
6,0.03,200,8.520180,34.748353
7,0.03,300,8.522732,34.764907
3,0.05,200,8.481597,34.772326
4,0.05,300,8.503683,34.792918
8,0.03,500,8.501513,34.806056
0,0.10,200,8.504571,34.837872
5,0.05,500,8.544730,34.879661
1,0.10,300,8.575111,34.920851
2,0.10,500,8.635026,35.050809


In [9]:
best_xgb = XGBRegressor(
    n_estimators=200,
    learning_rate=0.03,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

best_xgb.fit(X_train, y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [10]:
feature_importance = pd.DataFrame(
    {"Feature": model_features, "Importance": best_xgb.feature_importances_}
).sort_values("Importance", ascending=False)

feature_importance

,Feature,Importance
6,rolling_mean_28,0.310034
4,rolling_mean_7,0.153713
7,day_of_week,0.115047
3,lag_28,0.073751
0,lag_1,0.068128
2,lag_14,0.066111
1,lag_7,0.061979
8,month,0.057110
5,rolling_std_7,0.056766
9,week_of_year,0.025766


In [11]:
best_xgb_pred = best_xgb.predict(X_validation)

best_xgb_mae = mean_absolute_error(y_validation, best_xgb_pred)

best_xgb_rmse = mean_squared_error(y_validation, best_xgb_pred) ** 0.5

print("MAE:", best_xgb_mae)
print("RMSE:", best_xgb_rmse)

MAE: 8.520179709519613
RMSE: 34.748353228351014


In [13]:
zero_prediction = pd.Series(0, index=y_validation.index)

zero_mae = mean_absolute_error(y_validation, zero_prediction)

zero_rmse = mean_squared_error(y_validation, zero_prediction) ** 0.5

print("Zero Demand MAE:", zero_mae)
print("Zero Demand RMSE:", zero_rmse)

Zero Demand MAE: 7.705673161194634
Zero Demand RMSE: 37.840731405191534


In [14]:
naive_prediction = validation["lag_1"]

mae_naive = mean_absolute_error(y_validation, naive_prediction)

rmse_naive = mean_squared_error(y_validation, naive_prediction) ** 0.5

print("Naive MAE:", mae_naive)
print("Naive RMSE:", rmse_naive)

Naive MAE: 11.302534888424459
Naive RMSE: 49.119678713502225


In [15]:
results = pd.DataFrame(
    {
        "Model": [
            "Zero Demand",
            "Naive (Yesterday)",
            "7-Day Moving Average",
            "Linear Regression",
            "XGBoost",
        ],
        "MAE": [
            zero_mae,
            mae_naive,
            9.553136212849502,
            8.798927561624463,
            best_xgb_mae,
        ],
        "RMSE": [
            zero_rmse,
            rmse_naive,
            37.12059034442131,
            35.23593393973345,
            best_xgb_rmse,
        ],
    }
)

results

,Model,MAE,RMSE
0,Zero Demand,7.705673,37.840731
1,Naive (Yesterday),11.302535,49.119679
2,7-Day Moving Average,9.553136,37.120590
3,Linear Regression,8.798928,35.235934
4,XGBoost,8.520180,34.748353


### Baseline Model Comparison — Key Finding

The baseline comparison shows that the XGBoost model achieved the lowest RMSE (34.75) among the evaluated forecasting approaches, while the Zero Demand baseline achieved the lowest MAE (7.71).

This difference is important because the dataset contains highly intermittent and right-skewed SKU demand, with many zero-demand observations and occasional large demand spikes. A Zero Demand strategy can therefore achieve a deceptively low MAE by predicting zero for every observation, while performing worse on larger demand errors.

For this project, RMSE is prioritized as the primary model-selection metric because large forecasting errors can have greater operational consequences for inventory planning. MAE is retained as a complementary metric because it provides an interpretable measure of the average absolute forecasting error.

Based on the validation results, XGBoost is currently selected as the forecasting model for further error analysis and final evaluation.

**Validation results:**

- Zero Demand — MAE: 7.71, RMSE: 37.84
- Naive (Yesterday) — MAE: 11.30, RMSE: 49.12
- 7-Day Moving Average — MAE: 9.55, RMSE: 37.12
- Linear Regression — MAE: 8.80, RMSE: 35.24
- XGBoost — MAE: 8.52, RMSE: 34.75

The test set has not been used for model selection and will remain untouched until final evaluation.

In [16]:
xgb_results = pd.DataFrame({"Actual": y_validation.values, "Predicted": best_xgb_pred})

xgb_results["Error"] = xgb_results["Actual"] - xgb_results["Predicted"]

xgb_results["Absolute_Error"] = xgb_results["Error"].abs()

print("Overall MAE:", xgb_results["Absolute_Error"].mean())

print(
    "Zero-demand MAE:",
    xgb_results.loc[xgb_results["Actual"] == 0, "Absolute_Error"].mean(),
)

print(
    "Non-zero demand MAE:",
    xgb_results.loc[xgb_results["Actual"] > 0, "Absolute_Error"].mean(),
)

print(
    "High-demand MAE:",
    xgb_results.loc[xgb_results["Actual"] >= 20, "Absolute_Error"].mean(),
)

Overall MAE: 8.520179709519613
Zero-demand MAE: 3.815106080524305
Non-zero demand MAE: 17.330219660711162
High-demand MAE: 44.78512260449335


In [17]:
xgb_results.sort_values("Absolute_Error", ascending=False).head(10)

,Actual,Predicted,Error,Absolute_Error
6231,2868.0,17.721741,2850.278259,2850.278259
507,2004.0,1.644839,2002.355161,2002.355161
88709,1952.0,9.560822,1942.439178,1942.439178
88547,1944.0,9.691135,1934.308865,1934.308865
88601,1882.0,13.044557,1868.955443,1868.955443
25750,1815.0,22.948790,1792.051210,1792.051210
29819,1566.0,67.558884,1498.441116,1498.441116
86927,1496.0,14.577904,1481.422096,1481.422096
42797,1446.0,7.592764,1438.407236,1438.407236
115086,1583.0,149.719360,1433.280640,1433.280640


In [18]:
print("Mean Actual Demand:", y_validation.mean())
print("Mean Predicted Demand:", best_xgb_pred.mean())

print("Actual total demand:", y_validation.sum())

print("Predicted total demand:", best_xgb_pred.sum())

Mean Actual Demand: 7.705673161194634
Mean Predicted Demand: 7.074155
Actual total demand: 914386.0
Predicted total demand: 839447.5


### Model Bias — Underprediction of Demand Spikes

The XGBoost model produced a lower total predicted demand than the actual validation demand.

- Actual total demand: 914,386 units
- Predicted total demand: 839,447.5 units
- Aggregate underprediction: 74,938.5 units (~8.2%)

The error analysis showed that the largest forecasting errors occur on high-demand observations. This indicates that the current feature set and model configuration tend to be conservative when predicting unusually large demand spikes.

This limitation is important for inventory planning because systematic underprediction during demand spikes could contribute to insufficient replenishment decisions. The model will therefore be evaluated on the untouched test set before being used for downstream inventory recommendations.

In [19]:
X_test = test[model_features]
y_test = test["target"]

test_pred = best_xgb.predict(X_test)

test_mae = mean_absolute_error(y_test, test_pred)

test_rmse = mean_squared_error(y_test, test_pred) ** 0.5

print("Test MAE:", test_mae)
print("Test RMSE:", test_rmse)

Test MAE: 10.250109867507309
Test RMSE: 41.243261077540446


In [20]:
zero_test_pred = pd.Series(0, index=y_test.index)

zero_test_mae = mean_absolute_error(y_test, zero_test_pred)

zero_test_rmse = mean_squared_error(y_test, zero_test_pred) ** 0.5

In [21]:
naive_test_pred = test["lag_1"]

naive_test_mae = mean_absolute_error(y_test, naive_test_pred)

naive_test_rmse = mean_squared_error(y_test, naive_test_pred) ** 0.5

In [22]:
test_results = pd.DataFrame(
    {
        "Model": ["Zero Demand", "Naive (Yesterday)", "XGBoost"],
        "MAE": [zero_test_mae, naive_test_mae, test_mae],
        "RMSE": [zero_test_rmse, naive_test_rmse, test_rmse],
    }
)

test_results

,Model,MAE,RMSE
0,Zero Demand,10.454003,46.317395
1,Naive (Yesterday),14.156751,56.576767
2,XGBoost,10.250110,41.243261


### Final Model Evaluation on Unseen Test Data

After model selection and hyperparameter tuning on the training and validation periods, the selected XGBoost model was evaluated once on the untouched test period.

| Model | MAE | RMSE |
|---|---:|---:|
| Zero Demand | 10.45 | 46.32 |
| Naive (Yesterday) | 14.16 | 56.58 |
| XGBoost | 10.25 | 41.24 |

The XGBoost model achieved the lowest MAE and RMSE among the evaluated models on the unseen test data.

Test performance was weaker than validation performance (RMSE increased from 34.75 to 41.24), indicating some degradation when forecasting the later time period. This is expected to some extent in a temporal forecasting problem and highlights the importance of evaluating on a genuinely unseen period.

The model is therefore considered useful as a demand forecasting component, but its predictions should be interpreted alongside demand uncertainty and business inventory constraints rather than treated as exact future demand.

In [24]:
demand_model_data = pd.concat([train, validation, test], ignore_index=True)

In [25]:
print(demand_model_data.shape)
print(demand_model_data["StockCode"].nunique())

(713260, 15)
2435


In [26]:
sku_inventory_stats = (
    demand_model_data.groupby("StockCode")["Demand"]
    .agg(mean_daily_demand="mean", std_daily_demand="std")
    .reset_index()
)

sku_inventory_stats.head()

,StockCode,mean_daily_demand,std_daily_demand
0,10002,7.163636,25.385298
1,10125,3.249275,13.794649
2,10133,10.738095,22.532908
3,10135,5.281977,17.549567
4,11001,4.502924,22.700491


In [27]:
import numpy as np

lead_time = 7
z_score = 1.645

sku_inventory_stats["safety_stock"] = (
    z_score * sku_inventory_stats["std_daily_demand"] * np.sqrt(lead_time)
)

sku_inventory_stats.head()

,StockCode,mean_daily_demand,std_daily_demand,safety_stock
0,10002,7.163636,25.385298,110.483439
1,10125,3.249275,13.794649,60.037910
2,10133,10.738095,22.532908,98.069093
3,10135,5.281977,17.549567,76.380296
4,11001,4.502924,22.700491,98.798461


In [28]:
sku_inventory_stats["lead_time_demand"] = (
    sku_inventory_stats["mean_daily_demand"] * lead_time
)

In [29]:
sku_inventory_stats["reorder_point"] = (
    sku_inventory_stats["lead_time_demand"] + sku_inventory_stats["safety_stock"]
)

sku_inventory_stats.head()

,StockCode,mean_daily_demand,std_daily_demand,safety_stock,lead_time_demand,reorder_point
0,10002,7.163636,25.385298,110.483439,50.145455,160.628894
1,10125,3.249275,13.794649,60.037910,22.744928,82.782838
2,10133,10.738095,22.532908,98.069093,75.166667,173.235760
3,10135,5.281977,17.549567,76.380296,36.973837,113.354133
4,11001,4.502924,22.700491,98.798461,31.520468,130.318929


### Inventory Planning Assumptions

The original transaction dataset does not provide supplier lead times, inventory-on-hand levels, or target service levels. Therefore, the initial inventory decision layer uses transparent scenario assumptions:

- Lead time: 7 days
- Target service level: 95%
- Z-score: 1.645

Expected lead-time demand is estimated from historical mean daily demand. Safety stock is calculated from historical daily demand variability and the assumed lead time.

The Reorder Point (ROP) is defined as expected lead-time demand plus safety stock.

These values represent scenario-based inventory recommendations rather than actual company inventory policies and can be modified in the final application.

In [31]:
test_forecast = test[["StockCode", "Date", "target"]].copy()

test_forecast["predicted_demand"] = test_pred

test_forecast.head()

,StockCode,Date,target,predicted_demand
1090,10125,2011-10-13,0.0,4.738415
1091,10125,2011-10-14,0.0,1.679860
1092,10125,2011-10-15,0.0,3.251297
1093,10125,2011-10-16,20.0,4.234795
1094,10125,2011-10-17,0.0,6.453659


In [32]:
sku_forecast_summary = (
    test_forecast.groupby("StockCode")
    .agg(
        total_actual_demand=("target", "sum"),
        total_predicted_demand=("predicted_demand", "sum"),
        avg_predicted_daily_demand=("predicted_demand", "mean"),
    )
    .reset_index()
)

sku_forecast_summary.head()

,StockCode,total_actual_demand,total_predicted_demand,avg_predicted_daily_demand
0,10125,128.0,207.652573,3.643028
1,10135,309.0,280.865692,5.015459
2,11001,173.0,248.775360,4.606951
3,15034,720.0,1277.948608,22.820511
4,15036,1009.0,1325.686646,23.257660


In [33]:
sku_inventory = sku_inventory_stats.merge(
    sku_forecast_summary, on="StockCode", how="inner"
)

sku_inventory.head()

,StockCode,mean_daily_demand,std_daily_demand,safety_stock,lead_time_demand,reorder_point,total_actual_demand,total_predicted_demand,avg_predicted_daily_demand
0,10125,3.249275,13.794649,60.037910,22.744928,82.782838,128.0,207.652573,3.643028
1,10135,5.281977,17.549567,76.380296,36.973837,113.354133,309.0,280.865692,5.015459
2,11001,4.502924,22.700491,98.798461,31.520468,130.318929,173.0,248.775360,4.606951
3,15034,19.250000,102.069439,444.232829,134.750000,578.982829,720.0,1277.948608,22.820511
4,15036,68.790698,198.715312,864.860885,481.534884,1346.395769,1009.0,1325.686646,23.257660


In [34]:
sku_inventory["forecast_7_day_demand"] = sku_inventory["avg_predicted_daily_demand"] * 7

In [35]:
sku_inventory["forecast_reorder_point"] = (
    sku_inventory["forecast_7_day_demand"] + sku_inventory["safety_stock"]
)

sku_inventory[
    [
        "StockCode",
        "avg_predicted_daily_demand",
        "forecast_7_day_demand",
        "safety_stock",
        "forecast_reorder_point",
    ]
].head(10)

,StockCode,avg_predicted_daily_demand,forecast_7_day_demand,safety_stock,forecast_reorder_point
0,10125,3.643028,25.501192,60.037910,85.539102
1,10135,5.015459,35.108212,76.380296,111.488507
2,11001,4.606951,32.248657,98.798461,131.047118
3,15034,22.820511,159.743576,444.232829,603.976405
4,15036,23.257660,162.803619,864.860885,1027.664504
5,15039,6.044759,42.313316,110.911460,153.224777
6,16008,17.190050,120.330353,140.395156,260.725509
7,16011,6.657863,46.605042,80.376119,126.981161
8,16012,3.594366,25.160559,67.604024,92.764583
9,16014,22.780403,159.462830,1090.270197,1249.733027


In [36]:
recent_demand = (
    demand_model_data.sort_values(["StockCode", "Date"])
    .groupby("StockCode")
    .tail(14)
    .groupby("StockCode")["Demand"]
    .sum()
    .reset_index(name="simulated_inventory")
)

sku_inventory = sku_inventory.merge(recent_demand, on="StockCode", how="left")

sku_inventory.head()

,StockCode,mean_daily_demand,std_daily_demand,safety_stock,lead_time_demand,reorder_point,total_actual_demand,total_predicted_demand,avg_predicted_daily_demand,forecast_7_day_demand,forecast_reorder_point,simulated_inventory
0,10125,3.249275,13.794649,60.037910,22.744928,82.782838,128.0,207.652573,3.643028,25.501192,85.539102,6
1,10135,5.281977,17.549567,76.380296,36.973837,113.354133,309.0,280.865692,5.015459,35.108212,111.488507,116
2,11001,4.502924,22.700491,98.798461,31.520468,130.318929,173.0,248.775360,4.606951,32.248657,131.047118,75
3,15034,19.250000,102.069439,444.232829,134.750000,578.982829,720.0,1277.948608,22.820511,159.743576,603.976405,57
4,15036,68.790698,198.715312,864.860885,481.534884,1346.395769,1009.0,1325.686646,23.257660,162.803619,1027.664504,170


In [37]:
sku_inventory["reorder_needed"] = (
    sku_inventory["simulated_inventory"] < sku_inventory["forecast_reorder_point"]
)

sku_inventory[
    ["StockCode", "simulated_inventory", "forecast_reorder_point", "reorder_needed"]
].head(10)

,StockCode,simulated_inventory,forecast_reorder_point,reorder_needed
0,10125,6,85.539102,True
1,10135,116,111.488507,False
2,11001,75,131.047118,True
3,15034,57,603.976405,True
4,15036,170,1027.664504,True
5,15039,108,153.224777,True
6,16008,384,260.725509,False
7,16011,48,126.981161,True
8,16012,1,92.764583,True
9,16014,575,1249.733027,True


In [38]:
sku_inventory["inventory_gap"] = (
    sku_inventory["forecast_reorder_point"] - sku_inventory["simulated_inventory"]
)

sku_inventory["inventory_gap_pct"] = (
    sku_inventory["inventory_gap"] / sku_inventory["forecast_reorder_point"]
) * 100

In [39]:
sku_inventory["risk_level"] = pd.cut(
    sku_inventory["inventory_gap_pct"],
    bins=[-np.inf, 0, 25, 50, np.inf],
    labels=["Healthy", "Watch", "High", "Critical"],
)

In [40]:
sku_inventory[
    [
        "StockCode",
        "simulated_inventory",
        "forecast_reorder_point",
        "inventory_gap_pct",
        "risk_level",
    ]
].head(20)

,StockCode,simulated_inventory,forecast_reorder_point,inventory_gap_pct,risk_level
0,10125,6,85.539102,92.985664,Critical
1,10135,116,111.488507,-4.046599,Healthy
2,11001,75,131.047118,42.768677,High
3,15034,57,603.976405,90.562545,Critical
4,15036,170,1027.664504,83.457636,Critical
5,15039,108,153.224777,29.515316,High
6,16008,384,260.725509,-47.281331,Healthy
7,16011,48,126.981161,62.199117,Critical
8,16012,1,92.764583,98.922002,Critical
9,16014,575,1249.733027,53.990173,Critical


In [41]:
def recommend_action(risk):
    if risk == "Critical":
        return "Reorder immediately"
    elif risk == "High":
        return "Reorder soon"
    elif risk == "Watch":
        return "Monitor closely"
    else:
        return "No action"


sku_inventory["recommended_action"] = sku_inventory["risk_level"].map(recommend_action)

sku_inventory[
    [
        "StockCode",
        "simulated_inventory",
        "forecast_reorder_point",
        "risk_level",
        "recommended_action",
    ]
].head(20)

,StockCode,simulated_inventory,forecast_reorder_point,risk_level,recommended_action
0,10125,6,85.539102,Critical,Reorder immediately
1,10135,116,111.488507,Healthy,No action
2,11001,75,131.047118,High,Reorder soon
3,15034,57,603.976405,Critical,Reorder immediately
4,15036,170,1027.664504,Critical,Reorder immediately
5,15039,108,153.224777,High,Reorder soon
6,16008,384,260.725509,Healthy,No action
7,16011,48,126.981161,Critical,Reorder immediately
8,16012,1,92.764583,Critical,Reorder immediately
9,16014,575,1249.733027,Critical,Reorder immediately


In [42]:
sku_inventory["risk_level"].value_counts()

risk_level
Critical    877
Healthy     761
High        364
Watch       288
Name: count, dtype: int64

In [43]:
sku_inventory["recommended_action"].value_counts()

recommended_action
Reorder immediately    877
No action              761
Reorder soon           364
Monitor closely        288
Name: count, dtype: int64

In [44]:
decision_columns = [
    "StockCode",
    "mean_daily_demand",
    "std_daily_demand",
    "safety_stock",
    "forecast_7_day_demand",
    "forecast_reorder_point",
    "simulated_inventory",
    "inventory_gap",
    "inventory_gap_pct",
    "risk_level",
    "recommended_action",
]

decision_data = sku_inventory[decision_columns].copy()

decision_data.to_csv("../data/processed/stock_inventory_decisions.csv", index=False)

decision_data.head()

,StockCode,mean_daily_demand,std_daily_demand,safety_stock,forecast_7_day_demand,forecast_reorder_point,simulated_inventory,inventory_gap,inventory_gap_pct,risk_level,recommended_action
0,10125,3.249275,13.794649,60.037910,25.501192,85.539102,6,79.539102,92.985664,Critical,Reorder immediately
1,10135,5.281977,17.549567,76.380296,35.108212,111.488507,116,-4.511493,-4.046599,Healthy,No action
2,11001,4.502924,22.700491,98.798461,32.248657,131.047118,75,56.047118,42.768677,High,Reorder soon
3,15034,19.250000,102.069439,444.232829,159.743576,603.976405,57,546.976405,90.562545,Critical,Reorder immediately
4,15036,68.790698,198.715312,864.860885,162.803619,1027.664504,170,857.664504,83.457636,Critical,Reorder immediately


In [45]:
decision_data.isna().sum()

StockCode                 0
mean_daily_demand         0
std_daily_demand          0
safety_stock              0
forecast_7_day_demand     0
forecast_reorder_point    0
simulated_inventory       0
inventory_gap             0
inventory_gap_pct         0
risk_level                0
recommended_action        0
dtype: int64

In [46]:
decision_data.to_csv("../data/processed/stock_inventory_decisions.csv", index=False)